### 3.5.2. Gradient Method Framework


$$
x^{k+1}=x^k-\alpha^k D^k \nabla f(x^k),
\qquad
D^k \succ 0.
$$


**Explanation:**

Here, the term gradient method is used broadly: the search direction is related to the gradient and is chosen so that it is a descent direction whenever the current point is not stationary. The matrix $D^k$ defines the local scaling of the gradient step. With $D^k=I$, the method is steepest descent; with diagonal $D^k$, it becomes a scaled steepest descent method; with $D^k=(\nabla^2 f(x^k))^{-1}$ and a [positive definite Hessian](../../01_Foundations/01_Linear_Algebra/07_Theoretical_Linear_Algebra/17_positive_definite_matrix.ipynb), it becomes Newton's direction. The descent property follows from [positive definiteness](../../01_Foundations/01_Linear_Algebra/07_Theoretical_Linear_Algebra/17_positive_definite_matrix.ipynb), since $\nabla f(x^k)^{\mathsf T}D^k\nabla f(x^k)>0$ for a nonzero gradient.

The iterate $x^k$ is the current decision vector, $\alpha^k$ is the stepsize, and $D^k$ is a positive definite scaling matrix. The formula subtracts a scaled gradient, so the [matrix-vector product](../../01_Foundations/01_Linear_Algebra/03_Matrix/04_matrix_vector_multiplication.ipynb) $D^k\nabla f(x^k)$ determines the search direction.

This framework is important because it lets several algorithms be compared using the same notation. Instead of memorizing methods separately, the reader can ask what information is used to build $D^k$ and how expensive that information is to obtain. The later rate results then explain why better scaling can dramatically improve performance.

**Intuition:**

Steepest descent can zig-zag on elongated level sets because the negative gradient points across the valley rather than along it. Newton's method uses a quadratic model and curvature information to point more directly toward the minimizer near a nonsingular solution. The code cell contrasts an ordinary gradient step with a curvature-scaled step on an ill-conditioned quadratic.

<img src="../../Figures/030502_berts_fig_1_2_4_slow_steepest_descent.jpeg" alt="Bertsekas Figure 1.2.4: slow steepest descent on elongated level sets" style="width: 60%; height: auto;">

<img src="../../Figures/030502_berts_fig_1_2_5_newton_quadratic_model.jpeg" alt="Bertsekas Figure 1.2.5: Newton step from a quadratic approximation" style="width: 60%; height: auto;">


**Numerical Example:**

The elongated-level-set figure explains why scaling changes the step. Let
$$
Q=\begin{bmatrix}20&0\\0&1\end{bmatrix},\qquad f(x)=\frac{1}{2}x^{\mathsf T}Qx,\qquad x^0=\begin{bmatrix}1\\1\end{bmatrix}.
$$

Then
$$
\nabla f(x^0)=Qx^0=\begin{bmatrix}20\\1\end{bmatrix},
\qquad f(x^0)=\frac{1}{2}(20+1)=10.5.
$$

A steepest-descent step with $\alpha=0.08$ gives
$$
x^1_{\mathrm{sd}}=x^0-0.08\nabla f(x^0)
=\begin{bmatrix}1\\1\end{bmatrix}-0.08\begin{bmatrix}20\\1\end{bmatrix}
=\begin{bmatrix}-0.6\\0.92\end{bmatrix}.
$$

A scaled step with $D=Q^{-1}$ gives
$$
x^1_{\mathrm{scaled}}=x^0-Q^{-1}Qx^0=x^0-x^0=\begin{bmatrix}0\\0\end{bmatrix}.
$$

Thus curvature scaling moves directly to the minimizer for this quadratic, while the unscaled step only partially reduces the elongated error.


In [1]:
import sympy as sp

x_1, x_2 = sp.symbols("x_1 x_2")
decision_variable = sp.Matrix([x_1, x_2])
Q = sp.Matrix([[20, 0], [0, 1]])
objective = (decision_variable.T * Q * decision_variable)[0] / 2
gradient = sp.Matrix([sp.diff(objective, variable) for variable in decision_variable])
hessian = sp.hessian(objective, decision_variable)

initial_point = sp.Matrix([1, 1])
stepsize = sp.Rational(8, 100)
gradient_at_point = gradient.subs({x_1: initial_point[0], x_2: initial_point[1]})
steepest_point = initial_point - stepsize * gradient_at_point
scaled_point = initial_point - hessian.inv() * gradient_at_point

print("gradient ="); sp.pprint(gradient)
print("steepest_point ="); sp.pprint(steepest_point)
print("scaled_point ="); sp.pprint(scaled_point)

gradient =
⎡20⋅x₁⎤
⎢     ⎥
⎣ x₂  ⎦
steepest_point =
⎡-3/5⎤
⎢    ⎥
⎢ 23 ⎥
⎢ ── ⎥
⎣ 25 ⎦
scaled_point =
⎡0⎤
⎢ ⎥
⎣0⎦


**Equivalent `casadi` implementation:**

In [2]:
import casadi as ca

decision_variable = ca.SX.sym("x", 2)
Q = ca.DM([[20, 0], [0, 1]])
objective = 0.5 * ca.bilin(Q, decision_variable, decision_variable)
hessian, gradient_expr = ca.hessian(objective, decision_variable)
gradient_function = ca.Function("grad", [decision_variable], [gradient_expr])
hessian_function = ca.Function("hessian_eval", [decision_variable], [hessian])

initial_point = ca.DM([1, 1])
stepsize = 0.08
gradient = gradient_function(initial_point)
scaling_matrix = ca.inv(hessian_function(initial_point))
steepest_point = initial_point - stepsize * gradient
scaled_point = initial_point - scaling_matrix @ gradient

print("steepest_point =", steepest_point)
print("scaled_point =", scaled_point)

steepest_point = [-0.6, 0.92]
scaled_point = [0, 0]


**References:**

[📘 Bertsekas, D. P. (1999). *Nonlinear Programming* (2nd ed.), Chapter 1 "Unconstrained Optimization". Athena Scientific.](https://www.athenasc.com/nonlinbook.html)

---

[⬅️ Previous: Descent Directions](./01_descent_directions.ipynb) | [Next: Stepsize Rules ➡️](./03_stepsize_rules.ipynb)

---
